# Choose vs reach: why is Grassmann 26% behind attention?

The paper's layer averages six fixed offsets (1–16 tokens back) with equal weights. Attention instead **chooses** which earlier token to use, from the content, and **reaches** the whole block. This notebook separates the two with a 2×2 design on the paper's Plücker features:

| | equal-weight average | learned edge weights (graph attention over the offsets) |
|---|---|---|
| **offsets 1–16** (the paper's) | `grassmann` | `g-select` |
| **offsets 1, 2, 4, 16, 64, 127** (same count, whole block) | `g-dilated` | `g-both` |

The learned weights are a softmax over the 6 edges, scored from [z_t; z_{t−Δ}] (GATv2-style). The references are `swa` (chooses among the last 16) and `attention` (chooses among all).

2 seeds × 5 epochs at LR 1e-3, validation only. Pilot speeds before the speed fix: ~40–50 min on an A100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, json, subprocess, sys
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/choose_vs_reach.jsonl'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers
# drop any grassman modules imported earlier in this runtime, so the code just pulled is what runs
for _name in [k for k in sys.modules if k == 'grassman' or k.startswith('grassman.')]:
    del sys.modules[_name]

Only if you are **resuming this notebook** in a new runtime: set `RESTORE = True` to upload the `results.zip` it downloaded earlier, so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

In [ ]:
!python -m grassman.data --data-dir {DATA}

In [ ]:
# Run every model of this notebook on the same GPU type: precision differs (bf16 on A100/L4, fp16 on T4).
# COMPILE = True uses torch.compile for every model (fair across models); turn it on only if speed.ipynb
# showed compiled runs are faster on this GPU, since compiling costs time per run.
COMPILE = False

def train(mixer, seed, epochs=5, lr=1e-3, attn='', save=None, tag='pilot'):
    cmd = [sys.executable, '-m', 'grassman.train', '--mixer', mixer, '--seed', str(seed), '--lr', str(lr),
           '--epochs', str(epochs), '--tag', tag, '--data-dir', DATA, '--out', RUNS,
           '--skip-existing', '--no-test']
    if attn:
        cmd += ['--attn-layers', attn]
    if save:
        cmd += ['--save', save]
    if COMPILE:
        cmd += ['--compile']
    print('>>', mixer, f'attn@{attn}' if attn else '', 'seed', seed, flush=True)
    subprocess.run(cmd, cwd=REPO, check=True)

In [ ]:
MIXERS = ['grassmann', 'g-select', 'g-dilated', 'g-both', 'swa', 'attention']
SEEDS = [0, 1]
for seed in SEEDS:
    for m in MIXERS:
        train(m, seed)

## Results

Validation perplexity, lower is better. 'vs grassmann' is per seed; negative means better than the paper's layer.

In [ ]:
from grassman import analyze
rows = analyze.load(RUNS)
print(analyze.val_table(rows, 'grassmann'))

mean = lambda m: sum(r['best_val_ppl'] for r in rows if analyze.name(r) == m) / max(1, sum(analyze.name(r) == m for r in rows))
g, s, d, b = (mean(m) for m in ('grassmann', 'g-select', 'g-dilated', 'g-both'))
print(f"""
2x2 (mean validation perplexity)
                      equal weights   learned weights
  offsets 1-16        {g:13.1f}   {s:15.1f}
  offsets to 127      {d:13.1f}   {b:15.1f}

effect of choosing: {100 * ((s / g - 1) + (b / d - 1)) / 2:+.1f}%   (average over both reaches)
effect of reach:    {100 * ((d / g - 1) + (b / s - 1)) / 2:+.1f}%   (average over both weightings)
attention:          {mean('attention'):.1f}    swa: {mean('swa'):.1f}
gap to attention closed by g-both: {100 * (g - b) / (g - mean('attention')):.0f}%
""")

In [ ]:
import matplotlib.pyplot as plt
from grassman import analyze
rows = analyze.load(RUNS)
fig, ax = plt.subplots(figsize=(8, 4.5))
for r in rows:
    h = r['history']
    ax.plot([e['epoch'] for e in h], [e['val_ppl'] for e in h], ls='-' if r['seed'] == 0 else '--',
            label=f"{analyze.name(r)} s{r['seed']}")
ax.set_xlabel('epoch'); ax.set_ylabel('validation perplexity'); ax.grid(alpha=.3); ax.legend(fontsize=7, ncol=2)
plt.savefig(RUNS.replace('.jsonl', '_curves.png'), dpi=150, bbox_inches='tight'); plt.show()

## How to read it

- **Choosing helps, reach doesn't:** the gap is about selecting the right token. `g-select` should approach `swa`.
- **Reach helps, choosing doesn't:** the gap is about seeing far back. That argues for long filters (Hyena) over local pairs.
- **Both are needed:** `g-both` is the closest to attention, and the 'gap closed' line says how close a linear-cost layer gets.
- A difference smaller than the seed spread is noise. This is 5 epochs: a pilot, not a final result.

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')